# Ejercicio 05 · Motor recomendador reproducible

Notebook de evidencia del experimento LAB08/LAB09. Los datos MovieLens permanecen locales y no se redistribuyen. Resultados calculados con las semillas 17, 42 y 73; corte temporal por usuario.

In [2]:
from pathlib import Path
import pandas as pd, json
root=Path('..') if Path('../reports/main_results.csv').exists() else Path('.')
reports=root/'reports'
results=pd.read_csv(reports/'main_results.csv')
pareto=pd.read_csv(reports/'pareto_comparison.csv')
seed_runs=pd.read_csv(reports/'seed_metrics.csv')
auditoría=json.loads((reports/'data_audit.json').read_text(encoding='utf-8'))
print(f"MovieLens Latest Small: {auditoría['ratings']:,} ratings, {auditoría['movies']:,} películas, {auditoría['users']} usuarios")
print(f"Eventos temporales elegibles: {auditoría['eligible_test_events']}")
print(f"SHA-256 ZIP: {auditoría['sha256']}")

MovieLens Latest Small: 100,836 ratings, 9,742 películas, 610 usuarios
Eventos temporales elegibles: 587
SHA-256 ZIP: 696d65a3dfceac7c45750ad32df2c259311949efec81f0f144fdfb91ebc9e436


## Tabla principal

HitRate@10 usa un ítem retenido por usuario. Cobertura es la proporción de películas del catálogo que aparece en alguna lista. RMSE solo aplica a modelos de rating. El baseline de contenido expresa coincidencia de géneros, no satisfacción.

In [4]:
print(results.round(4).to_string(index=False))

           method   rmse  hit_rate_at_10  hit_rate_sd  catalog_coverage  fit_seconds  model_parameters  evaluated_users
    collaborative 1.0272          0.0312       0.0055            0.0737       9.7511            206220              587
          content    NaN          0.0000       0.0000            0.1495       0.0000                 0              587
hybrid_alpha_0.25 1.0272          0.0290       0.0034            0.1038       9.7511            206220              587
hybrid_alpha_0.75 1.0272          0.0312       0.0055            0.0760       9.7511            206220              587
       popularity    NaN          0.0017       0.0000            0.0013       0.0000                 0              587


## Comparación Pareto del híbrido

Se maximiza HitRate@10 y cobertura, y se minimiza tiempo de entrenamiento y número de parámetros. La decisión debe considerar el objetivo: alpha alto prioriza aciertos; alpha bajo amplía cobertura.

In [6]:
print(pareto.round(4).to_string(index=False))

 factors  epochs  alpha   rmse  rmse_sd  hit_rate_at_10  hit_rate_sd  catalog_coverage  fit_seconds  model_parameters  evaluated_users  pareto_efficient
      20      12   0.25 1.0272   0.0056          0.0290       0.0034            0.1038       9.7511            206220              587              True
      20      12   0.75 1.0272   0.0056          0.0312       0.0055            0.0760       9.7511            206220              587              True


## Interpretación y límites

En esta corrida alpha=0.75 igualó el HitRate@10 colaborativo con cobertura ligeramente mayor; alpha=0.25 obtuvo mayor cobertura, con una pequeña reducción de aciertos. Ambos pesos son Pareto eficientes. Se elige alpha=0.75 si el objetivo primario es recuperar el ítem retenido, y se conserva alpha=0.25 como alternativa de mayor exposición de catálogo. Solo se evalúan 587 usuarios/ítems conocidos por entrenamiento: cold start queda fuera de la métrica. Los resultados no miden satisfacción real ni significancia estadística. MovieLens contiene ratings autoseleccionados, carece de demografía y puede cambiar entre descargas.